In [1]:
import gc
from itertools import combinations
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.base import clone
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
MODEL_DIR = ROOT / "models" / "randomized_search"
OUTPUT_DIR = ROOT / "results" / "friedman_test"
FOLD_RESULTS_DIR = OUTPUT_DIR / "folds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLD_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_REPEATS = 30
N_JOBS = -1
ALPHA = 0.05
PRIMARY_METRIC = "f1_macro"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

MODEL_ORDER = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

TRAINING_SOURCES_TO_RUN = list(TRAINING_CONFIGS)
MODELS_TO_RUN = MODEL_ORDER.copy()

#pos_label=0 benigno ; pos_label=1 ataque.
#zero_division=0. se a métrica envolver uma divisão por zero, registre o resultado como zero.
scoring = {
    "precision_attack": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_attack": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_attack": make_scorer(f1_score, pos_label=1, zero_division=0),
    "precision_benign": make_scorer(precision_score, pos_label=0, zero_division=0),
    "recall_benign": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_benign": make_scorer(f1_score, pos_label=0, zero_division=0),
    "f1_macro": make_scorer(
        f1_score,
        average="macro",
        labels=[0, 1],
        zero_division=0,
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}


In [3]:
model_files = {
    (model_name, training_source): (
        MODEL_DIR / model_name / f"{training_source}.joblib"
    )
    for model_name in MODELS_TO_RUN
    for training_source in TRAINING_SOURCES_TO_RUN
}

In [4]:
# load na combinação (modelo, base)| recuperar colunas | limpar objetos
reference_pipeline = joblib.load(model_files[(MODEL_ORDER[0], TRAINING_SOURCES_TO_RUN[0])])
preprocessor = reference_pipeline.named_steps["preprocessor"]
numerical_features = list(preprocessor.transformers[0][2])
categorical_features = list(preprocessor.transformers[1][2])
feature_columns = numerical_features + categorical_features
del reference_pipeline, preprocessor

repeated_cross_validation = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

def load_training_data(dataset_names):
    columns_to_load = feature_columns + ["label_binary", "dataset_name"]
    dataframes = [
        pd.read_parquet(
            DATA_DIR / f"{dataset_name}_train.parquet",
            columns=columns_to_load,
        )
        for dataset_name in dataset_names
    ]
    training_df = pd.concat(dataframes, ignore_index=True)
    features = training_df[feature_columns]
    target = training_df["label_binary"].eq("ATTACK").astype("int8")

    #Estratificação por base e classe
    stratification_labels = (
        training_df["dataset_name"].astype("string")
        + "__"
        + target.astype("string")
    )
    # gera e fixa 30 repetições de 5 folds estratificados (150 pares de treino e teste).
    cv_splits = list(
        repeated_cross_validation.split(features, stratification_labels)
    )
    return features, target, cv_splits

#monta o caminho do csv com os 150 resultados de cada combinação
def fold_result_path(model_name, training_source):
    model_output_dir = FOLD_RESULTS_DIR / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)
    return model_output_dir / f"{training_source}.csv"

# verifica se o resultado está completo para evitar uma nova execução.
def result_is_complete(result_path):
    if not result_path.exists():
        return False
    result = pd.read_csv(result_path)
    expected_rows = N_SPLITS * N_REPEATS
    required_columns = {
        "model",
        "training_source",
        "repeat",
        "fold",
        "n_splits",
        "n_repeats",
        "random_state",
        PRIMARY_METRIC,
    }
    if len(result) != expected_rows:
        return False
    if not required_columns.issubset(result.columns):
        return False
    # confirma que o arquivo corresponde à configuração atual e não possui resultados inválidos
    return (
        result["n_splits"].eq(N_SPLITS).all()
        and result["n_repeats"].eq(N_REPEATS).all()
        and result["random_state"].eq(RANDOM_STATE).all()
        and not result[["repeat", "fold"]].duplicated().any()
        and not result[PRIMARY_METRIC].isna().any()
    )

#avalia um modelo com seus melhores hiperparâmetros nas 30 repetições do k-fold
def evaluate_selected_model(model_name, training_source, features, target, cv_splits):

    #clone mantém a configuração do modelo, mas remove o treinamento anterior
    selected_pipeline = clone(joblib.load(model_files[(model_name, training_source)]))
    evaluation_start = perf_counter()
    # executa os 150 pares de treino e teste
    cv_results = cross_validate(
        selected_pipeline,
        features,
        target,
        scoring=scoring,
        cv=cv_splits,
        n_jobs=N_JOBS,
        return_train_score=False,
        error_score="raise",
    )
    elapsed_seconds = perf_counter() - evaluation_start
    #organiza cada resultado pelo número da repetição e do fold
    result = pd.DataFrame({
        "model": model_name,
        "training_source": training_source,
        "repeat": np.repeat(np.arange(1, N_REPEATS + 1), N_SPLITS),
        "fold": np.tile(np.arange(1, N_SPLITS + 1), N_REPEATS),
        "n_splits": N_SPLITS,
        "n_repeats": N_REPEATS,
        "random_state": RANDOM_STATE,
        "fit_time_seconds": cv_results["fit_time"],
        "score_time_seconds": cv_results["score_time"],
    })
    # adiciona ao resultado as métricas calculadas em cada fold
    for metric_name in scoring:
        result[metric_name] = cv_results[f"test_{metric_name}"]
    result["evaluation_elapsed_seconds"] = elapsed_seconds
    return result


In [5]:
# chama as funções da CV
total_start = perf_counter()
completed_experiments = 0
total_experiments = len(TRAINING_SOURCES_TO_RUN) * len(MODELS_TO_RUN)

for training_source in TRAINING_SOURCES_TO_RUN:
    pending_models = [
        model_name
        for model_name in MODELS_TO_RUN
        if not result_is_complete(fold_result_path(model_name, training_source))
    ]

    if pending_models:
        features, target, cv_splits = load_training_data(
            TRAINING_CONFIGS[training_source]
        )

        for model_name in MODELS_TO_RUN:
            result_path = fold_result_path(model_name, training_source)
            completed_experiments += 1

            if result_is_complete(result_path):
                print(
                    f"{completed_experiments:02d}/{total_experiments} - "
                    f"{model_name} - {training_source} - já concluído"
                )
                continue

            result = evaluate_selected_model(
                model_name,
                training_source,
                features,
                target,
                cv_splits,
            )
            result.to_csv(result_path, index=False)
            elapsed_minutes = result["evaluation_elapsed_seconds"].iloc[0] / 60
            print(
                f"{completed_experiments:02d}/{total_experiments} - "
                f"{model_name} - {training_source} - "
                f"{elapsed_minutes:.2f} min"
            )

        del features, target, cv_splits
        gc.collect()
    else:
        completed_experiments += len(MODELS_TO_RUN)
        print(f"{training_source} - todos os modelos já concluídos")

total_elapsed_seconds = perf_counter() - total_start
print(f"tempo desta execução: {total_elapsed_seconds / 60:.2f} min")


01/30 - decision_tree - cicids2017 - 0.66 min
02/30 - random_forest - cicids2017 - 17.02 min
03/30 - xgboost - cicids2017 - 2.25 min
04/30 - linear_svm - cicids2017 - 4.00 min
05/30 - mlp - cicids2017 - 11.03 min
06/30 - decision_tree - unsw_nb15 - 0.55 min
07/30 - random_forest - unsw_nb15 - 16.87 min
08/30 - xgboost - unsw_nb15 - 2.23 min
09/30 - linear_svm - unsw_nb15 - 2.90 min
10/30 - mlp - unsw_nb15 - 17.49 min
11/30 - decision_tree - iot23 - 0.31 min
12/30 - random_forest - iot23 - 11.32 min
13/30 - xgboost - iot23 - 1.80 min
14/30 - linear_svm - iot23 - 1.54 min
15/30 - mlp - iot23 - 6.06 min
16/30 - decision_tree - cicids2017_unsw_nb15 - 1.44 min
17/30 - random_forest - cicids2017_unsw_nb15 - 45.80 min
18/30 - xgboost - cicids2017_unsw_nb15 - 5.08 min
19/30 - linear_svm - cicids2017_unsw_nb15 - 7.24 min
20/30 - mlp - cicids2017_unsw_nb15 - 55.36 min
21/30 - decision_tree - cicids2017_iot23 - 1.21 min
22/30 - random_forest - cicids2017_iot23 - 25.98 min
23/30 - xgboost - cicids

In [5]:
# dfs carregados dos csvs individuais de cada modelo e origem
fold_frames = [
    pd.read_csv(fold_result_path(model_name, training_source))
    for training_source in TRAINING_SOURCES_TO_RUN
    for model_name in MODELS_TO_RUN
]

# concatena e organiza os 30 dfs com os resultados dos folds
fold_results = pd.concat(fold_frames, ignore_index=True)
fold_results = fold_results.sort_values(
    ["training_source", "model", "repeat", "fold"]
).reset_index(drop=True)

# define como agregar os 5 folds: média das métricas e soma dos tempos
aggregation = {metric_name: "mean" for metric_name in scoring}
aggregation.update({
    "fit_time_seconds": "sum",
    "score_time_seconds": "sum",
})

# agrupa os 5 folds de cada repetição por origem e modelo
# aplica a média às métricas e a soma aos tempos, reduzindo os 4.500
# resultados por fold a 900 resultados por repetição
repeat_results = (
    fold_results.groupby(
        ["training_source", "model", "repeat"],
        as_index=False,
    )
    .agg(aggregation)
)
# salva 900 resultados: 30 repetições para cada combinação de origem e modelo
repeat_results.to_csv(OUTPUT_DIR / "repeat_results.csv", index=False)


In [6]:
# para cada origem, Friedman compara os 5 modelos em 30 repetições pareadas
# cada valor comparado é o F1-macro médio dos 5 folds daquela repetição
# o teste transforma esses valores em rankings e verifica se os desempenhos são equivalentes

# armazena os resultados e as matrizes de cada origem
friedman_rows = []
friedman_pivots = {}

for training_source in TRAINING_SOURCES_TO_RUN:
    # seleciona os 150 resultados da origem atual
    source_results = repeat_results.loc[
        repeat_results["training_source"].eq(training_source)
    ]
    # organiza as 30 repetições nas linhas e os 5 modelos nas colunas
    pivot = source_results.pivot(
        index="repeat",
        columns="model",
        values=PRIMARY_METRIC,
    ).reindex(columns=MODEL_ORDER)

    # confirma que a matriz está completa
    assert pivot.shape == (N_REPEATS, len(MODEL_ORDER))
    assert not pivot.isna().any().any()

    # executa Friedman com as 30 médias de cada modelo
    statistic, p_value = friedmanchisquare(
        *[pivot[model_name].to_numpy() for model_name in MODEL_ORDER]
    )
    # calcula a posição média de cada modelo nas 30 repetições
    ranks = pivot.rank(axis=1, ascending=False, method="average")
    row = {
        "training_source": training_source,
        "statistic": statistic,
        "p_value": p_value,
    }
    for model_name in MODEL_ORDER:
        row[f"mean_rank_{model_name}"] = ranks[model_name].mean()
    friedman_rows.append(row)
    friedman_pivots[training_source] = pivot

friedman_results = pd.DataFrame(friedman_rows)
friedman_results.to_csv(OUTPUT_DIR / "friedman_results.csv", index=False)

display(friedman_results[["training_source", "statistic", "p_value"]])


,training_source,statistic,p_value
0,cicids2017,117.840000,1.545044e-24
1,unsw_nb15,118.506667,1.113231e-24
2,iot23,114.186667,9.306935e-24
3,cicids2017_unsw_nb15,120.000000,5.341472e-25
4,cicids2017_iot23,120.000000,5.341472e-25
5,unsw_nb15_iot23,115.306667,5.367452e-24


In [7]:
# corrige os p-values das comparações múltiplas do pós-hoc
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running_maximum = 0.0

    for position, original_index in enumerate(order):
        multiplier = len(p_values) - position
        corrected_value = multiplier * p_values[original_index]
        running_maximum = max(running_maximum, corrected_value)
        adjusted[original_index] = min(running_maximum, 1.0)

    return adjusted


posthoc_rows = []

for friedman_result in friedman_results.itertuples(index=False):
    if friedman_result.p_value >= ALPHA:
        continue

    pivot = friedman_pivots[friedman_result.training_source]
    source_rows = []

    for model_a, model_b in combinations(MODEL_ORDER, 2):
        values_a = pivot[model_a].to_numpy()
        values_b = pivot[model_b].to_numpy()

        if np.allclose(values_a, values_b):
            statistic = 0.0
            p_value = 1.0
        else:
            statistic, p_value = wilcoxon(
                values_a,
                values_b,
                alternative="two-sided",
                method="auto",
            )

        source_rows.append({
            "training_source": friedman_result.training_source,
            "metric": PRIMARY_METRIC,
            "model_a": model_a,
            "model_b": model_b,
            "mean_a": values_a.mean(),
            "mean_b": values_b.mean(),
            "mean_difference_a_minus_b": values_a.mean() - values_b.mean(),
            "statistic": statistic,
            "p_value": p_value,
        })

    adjusted_p_values = holm_adjust([row["p_value"] for row in source_rows])
    for row, adjusted_p_value in zip(source_rows, adjusted_p_values):
        row["p_value_holm"] = adjusted_p_value
        row["significant"] = adjusted_p_value < ALPHA
        posthoc_rows.append(row)

posthoc_columns = [
    "training_source",
    "metric",
    "model_a",
    "model_b",
    "mean_a",
    "mean_b",
    "mean_difference_a_minus_b",
    "statistic",
    "p_value",
    "p_value_holm",
    "significant",
]
posthoc_results = pd.DataFrame(posthoc_rows, columns=posthoc_columns)
posthoc_results.to_csv(OUTPUT_DIR / "posthoc_wilcoxon_holm.csv", index=False)

if posthoc_results.empty:
    print("nenhum pós-hoc foi necessário")
else:
    display(posthoc_results)


,training_source,metric,model_a,model_b,mean_a,mean_b,mean_difference_a_minus_b,statistic,p_value,p_value_holm,significant
0,cicids2017,f1_macro,decision_tree,random_forest,0.990408,0.992492,-0.002084,0.0,1.862645e-09,1.862645e-08,True
1,cicids2017,f1_macro,decision_tree,xgboost,0.990408,0.991770,-0.001362,0.0,1.862645e-09,1.862645e-08,True
2,cicids2017,f1_macro,decision_tree,linear_svm,0.990408,0.903022,0.087386,0.0,1.862645e-09,1.862645e-08,True
3,cicids2017,f1_macro,decision_tree,mlp,0.990408,0.990556,-0.000147,51.0,6.286614e-05,6.286614e-05,True
4,cicids2017,f1_macro,random_forest,xgboost,0.992492,0.991770,0.000722,0.0,1.862645e-09,1.862645e-08,True
5,cicids2017,f1_macro,random_forest,linear_svm,0.992492,0.903022,0.089470,0.0,1.862645e-09,1.862645e-08,True
6,cicids2017,f1_macro,random_forest,mlp,0.992492,0.990556,0.001936,0.0,1.862645e-09,1.862645e-08,True
7,cicids2017,f1_macro,xgboost,linear_svm,0.991770,0.903022,0.088748,0.0,1.862645e-09,1.862645e-08,True
8,cicids2017,f1_macro,xgboost,mlp,0.991770,0.990556,0.001214,0.0,1.862645e-09,1.862645e-08,True
9,cicids2017,f1_macro,linear_svm,mlp,0.903022,0.990556,-0.087534,0.0,1.862645e-09,1.862645e-08,True


In [12]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

RESULTS_ROOT = Path(r"C:\tcc\network-ids-ml-generalization\results\friedman_test")
PRESENTATION_DIR = RESULTS_ROOT / "presentation"
PRESENTATION_DIR.mkdir(parents=True, exist_ok=True)

REPEAT_RESULTS_FILE = RESULTS_ROOT / "repeat_results.csv"
FRIEDMAN_RESULTS_FILE = RESULTS_ROOT / "friedman_results.csv"
POSTHOC_RESULTS_FILE = RESULTS_ROOT / "posthoc_wilcoxon_holm.csv"

for result_file in [
    REPEAT_RESULTS_FILE,
    FRIEDMAN_RESULTS_FILE,
    POSTHOC_RESULTS_FILE,
]:
    if not result_file.exists():
        raise FileNotFoundError(f"Resultado calculado não encontrado: {result_file}")

saved_repeat_results = pd.read_csv(REPEAT_RESULTS_FILE)
saved_friedman_results = pd.read_csv(FRIEDMAN_RESULTS_FILE)
saved_posthoc_results = pd.read_csv(POSTHOC_RESULTS_FILE)

assert len(saved_repeat_results) == 900
assert len(saved_friedman_results) == 6
assert len(saved_posthoc_results) == 60
assert saved_repeat_results.groupby(["training_source", "model"]).size().eq(30).all()

MODEL_ORDER = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
TRAINING_SOURCE_ORDER = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
    "cicids2017_unsw_nb15",
    "cicids2017_iot23",
    "unsw_nb15_iot23",
]
MODEL_LABELS = {
    "decision_tree": "Árvore de decisão",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "linear_svm": "SVM linear",
    "mlp": "MLP",
}
SOURCE_LABELS = {
    "cicids2017": "CICIDS2017",
    "unsw_nb15": "UNSW-NB15",
    "iot23": "IoT-23",
    "cicids2017_unsw_nb15": "CICIDS2017 + UNSW-NB15",
    "cicids2017_iot23": "CICIDS2017 + IoT-23",
    "unsw_nb15_iot23": "UNSW-NB15 + IoT-23",
}
MODEL_COLORS = {
    "Árvore de decisão": "#4C78A8",
    "Random Forest": "#F58518",
    "XGBoost": "#54A24B",
    "SVM linear": "#E45756",
    "MLP": "#B279A2",
}

loaded_results = pd.DataFrame({
    "arquivo": [
        REPEAT_RESULTS_FILE.name,
        FRIEDMAN_RESULTS_FILE.name,
        POSTHOC_RESULTS_FILE.name,
    ],
    "linhas": [
        len(saved_repeat_results),
        len(saved_friedman_results),
        len(saved_posthoc_results),
    ],
    "status": "carregado",
})
display(loaded_results)


,arquivo,linhas,status
0,repeat_results.csv,900,carregado
1,friedman_results.csv,6,carregado
2,posthoc_wilcoxon_holm.csv,60,carregado


In [11]:
boxplot_data = saved_repeat_results.copy()
boxplot_data["Modelo"] = boxplot_data["model"].map(MODEL_LABELS)
boxplot_data["Origem do treinamento"] = (
    boxplot_data["training_source"].map(SOURCE_LABELS)
)

boxplot_figure = px.box(
    boxplot_data,
    x="Modelo",
    y="f1_macro",
    color="Modelo",
    facet_col="Origem do treinamento",
    facet_col_wrap=2,
    points=False,
    color_discrete_map=MODEL_COLORS,
    category_orders={
        "Modelo": [MODEL_LABELS[model] for model in MODEL_ORDER],
        "Origem do treinamento": [
            SOURCE_LABELS[source] for source in TRAINING_SOURCE_ORDER
        ],
    },
    labels={"f1_macro": "F1-macro"},
    title=(
        "Distribuição do F1-macro nas 30 repetições da validação cruzada"
        "<br><sup>Cada boxplot resume 30 valores; cada valor é a média dos cinco folds de uma repetição</sup>"
    ),
)
boxplot_figure.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.split("=")[-1]
    )
)
boxplot_figure.update_traces(line={"width": 1.5})
boxplot_figure.update_yaxes(range=[0.80, 1.001], matches="y")
boxplot_figure.update_xaxes(tickangle=-20, showticklabels=True)
boxplot_figure.update_layout(
    template="plotly_white",
    width=1500,
    height=1150,
    showlegend=False,
)
boxplot_figure.write_html(
    PRESENTATION_DIR / "f1_macro_repeated_cv_boxplots.html",
    include_plotlyjs="cdn",
)
display(boxplot_figure)


In [14]:
distribution_summary = (
    saved_repeat_results.groupby(["training_source", "model"], as_index=False)
    .agg(
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
        f1_macro_median=("f1_macro", "median"),
        f1_macro_min=("f1_macro", "min"),
        f1_macro_max=("f1_macro", "max"),
    )
)
distribution_summary["position_by_mean"] = (
    distribution_summary.groupby("training_source")["f1_macro_mean"]
    .rank(method="min", ascending=False)
    .astype(int)
)
distribution_summary["training_source_label"] = (
    distribution_summary["training_source"].map(SOURCE_LABELS)
)
distribution_summary["model_label"] = distribution_summary["model"].map(MODEL_LABELS)
distribution_summary = distribution_summary.sort_values(
    ["training_source", "position_by_mean"]
)
mean_std_table = distribution_summary.copy()
mean_std_table["f1_macro_mean_std"] = mean_std_table.apply(
    lambda row: f"{row['f1_macro_mean']:.6f} ± {row['f1_macro_std']:.6f}",
    axis=1,
)
mean_std_pivot = (
    mean_std_table.pivot(
        index="training_source_label",
        columns="model_label",
        values="f1_macro_mean_std",
    )
    .reindex(
        index=[SOURCE_LABELS[source] for source in TRAINING_SOURCE_ORDER],
        columns=[MODEL_LABELS[model] for model in MODEL_ORDER],
    )
)
mean_std_pivot.to_csv(PRESENTATION_DIR / "f1_macro_mean_std.csv")

friedman_presentation = saved_friedman_results[[
    "training_source",
    "statistic",
    "p_value",
]].copy()
friedman_presentation.insert(
    1,
    "training_source_label",
    friedman_presentation["training_source"].map(SOURCE_LABELS),
)
friedman_presentation.to_csv(
    PRESENTATION_DIR / "friedman_test_summary.csv",
    index=False,
)

rank_rows = []
for result in saved_friedman_results.itertuples(index=False):
    for model in MODEL_ORDER:
        rank_rows.append({
            "training_source": result.training_source,
            "model": model,
            "mean_rank": getattr(result, f"mean_rank_{model}"),
        })
rank_summary = pd.DataFrame(rank_rows)
rank_summary["training_source_label"] = rank_summary["training_source"].map(SOURCE_LABELS)
rank_summary["model_label"] = rank_summary["model"].map(MODEL_LABELS)
rank_summary.to_csv(PRESENTATION_DIR / "mean_rank_summary.csv", index=False)

# organiza os p-valores ajustados dos 10 pares nas 6 origens de treinamento
posthoc_presentation = saved_posthoc_results.copy()
posthoc_presentation["pair"] = (
    posthoc_presentation["model_a"].map(MODEL_LABELS)
    + " × "
    + posthoc_presentation["model_b"].map(MODEL_LABELS)
)
posthoc_presentation["training_source_label"] = (
    posthoc_presentation["training_source"].map(SOURCE_LABELS)
)
pair_order = posthoc_presentation.loc[
    posthoc_presentation["training_source"].eq(TRAINING_SOURCE_ORDER[0]),
    "pair",
].tolist()
posthoc_pivot = (
    posthoc_presentation.pivot(
        index="pair",
        columns="training_source_label",
        values="p_value_holm",
    )
    .reindex(
        index=pair_order,
        columns=[SOURCE_LABELS[source] for source in TRAINING_SOURCE_ORDER],
    )
)
posthoc_pivot.index.name = "Comparação"
posthoc_pivot.columns.name = None
posthoc_pivot.to_csv(
    PRESENTATION_DIR / "posthoc_wilcoxon_holm_summary.csv"
)
display(posthoc_pivot)


,CICIDS2017,UNSW-NB15,IoT-23,CICIDS2017 + UNSW-NB15,CICIDS2017 + IoT-23,UNSW-NB15 + IoT-23
Comparação,,,,,,
Árvore de decisão × Random Forest,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
Árvore de decisão × XGBoost,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
Árvore de decisão × SVM linear,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
Árvore de decisão × MLP,6.286614e-05,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
Random Forest × XGBoost,1.862645e-08,1.862645e-08,2.560090e-03,1.862645e-08,1.862645e-08,6.639536e-03
Random Forest × SVM linear,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
Random Forest × MLP,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
XGBoost × SVM linear,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
XGBoost × MLP,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08,1.862645e-08
